# Financial Sentiment Analysis: AAPL News Using MLlib (PySpark)

This notebook analyzes Apple Inc. (AAPL) news sentiment using Spark MLlib and PySpark, following a Bronze → Silver → Gold data pipeline architecture.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.ml.feature import Tokenizer, StopWordsRemover
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Set JAVA_HOME if not already set
if 'JAVA_HOME' not in os.environ:
    java_base = r'C:\Program Files\Java'
    if os.path.exists(java_base):
        for item in os.listdir(java_base):
            if 'jdk' in item.lower():
                os.environ['JAVA_HOME'] = os.path.join(java_base, item)
                break

# Initialize Spark
spark = SparkSession.builder \
    .appName("FinancialSentimentAnalysis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")


## 1. Data Ingestion (Bronze Layer)


In [ ]:
# Load raw data (Bronze Layer)
df_news_bronze_full = spark.read.json("../data/aapl_news.json", multiLine=True)
df_price_bronze_full = spark.read.json("../data/aapl_price.json", multiLine=True)

# Ingest only 20% of the data for analysis to reduce resource usage
df_news_bronze = df_news_bronze_full.sample(fraction=0.2, seed=42)
df_price_bronze = df_price_bronze_full.sample(fraction=0.2, seed=42)

print("Using 20% of the raw data for analysis.")

# Inspect structure on the sampled data
print("\nNews data schema (sample):")
df_news_bronze.printSchema()
print("\nNews sample (sampled data):")
df_news_bronze.show(3, truncate=False)

print("\nPrice data schema (sample):")
df_price_bronze.printSchema()
print("\nPrice sample (sampled data):")
df_price_bronze.show(3, truncate=False)


## 2. Data Curation (Silver Layer)


In [ ]:
# Clean news data - handle two different formats
# Check which format we have by examining columns and timestamp format
news_columns = df_news_bronze.columns
print(f"Available news columns: {news_columns}")

# Detect format based on available columns
has_created = "created" in news_columns
has_created_at = "created_at" in news_columns
has_timestamp = "timestamp" in news_columns

# Select base columns (handle both formats)
if has_created:
    date_col = "created"
elif has_created_at:
    date_col = "created_at"
elif has_timestamp:
    date_col = "timestamp"
else:
    date_col = None

# Build base selection
base_cols = []
if "id" in news_columns:
    base_cols.append("id")
if "title" in news_columns:
    base_cols.append("title")
if "body" in news_columns:
    base_cols.append("body")
if "content" in news_columns:  # Alternative format might use "content"
    base_cols.append("content")
if date_col:
    base_cols.append(date_col)

df_news_silver = df_news_bronze.select(*base_cols)

# Handle timestamp conversion for multiple formats
# Format 1: ISO timestamp (yyyy-MM-dd'T'HH:mm:ss.SSS'Z')
# Format 2: RFC 2822 format (e.g., 'Fri, 02 Jan 2015 11:12:32 -0400')
# Format 3: Other common formats

from datetime import datetime
from email.utils import parsedate_tz, mktime_tz
from pyspark.sql.functions import udf
from pyspark.sql.types import TimestampType

def safe_news_timestamp_convert(ts):
    """Safely convert news timestamp string to datetime"""
    if ts is None:
        return None
    ts_str = str(ts).strip()
    
    # Try RFC 2822 format first (e.g., 'Fri, 02 Jan 2015 11:12:32 -0400')
    try:
        time_tuple = parsedate_tz(ts_str)
        if time_tuple:
            timestamp = mktime_tz(time_tuple)
            return datetime.fromtimestamp(timestamp)
    except:
        pass
    
    # Try ISO formats
    iso_formats = [
        "%Y-%m-%dT%H:%M:%S.%fZ",
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d"
    ]
    for fmt in iso_formats:
        try:
            dt = datetime.strptime(ts_str, fmt)
            return dt
        except:
            continue
    
    # Try numeric (Unix timestamp)
    try:
        ts_num = float(ts_str)
        if ts_num > 1e12:  # milliseconds
            return datetime.fromtimestamp(ts_num / 1000)
        else:  # seconds
            return datetime.fromtimestamp(ts_num)
    except:
        return None

# Register UDF
safe_news_timestamp_udf = udf(safe_news_timestamp_convert, TimestampType())

# Apply conversion
if date_col:
    df_news_silver = df_news_silver.withColumn(
        "created_ts",
        safe_news_timestamp_udf(col(date_col))
    )
else:
    df_news_silver = df_news_silver.withColumn("created_ts", lit(None).cast(TimestampType()))

# Extract date
df_news_silver = df_news_silver.withColumn("date", to_date(col("created_ts")))

# Combine title and body/content for sentiment analysis
if "body" in df_news_silver.columns and "content" in df_news_silver.columns:
    df_news_silver = df_news_silver.withColumn(
        "text",
        concat_ws(" ",
            coalesce(col("title"), lit("")),
            coalesce(col("body"), lit("")),
            coalesce(col("content"), lit(""))
        )
    )
elif "body" in df_news_silver.columns:
    df_news_silver = df_news_silver.withColumn(
        "text",
        concat_ws(" ", coalesce(col("title"), lit("")), coalesce(col("body"), lit("")))
    )
elif "content" in df_news_silver.columns:
    df_news_silver = df_news_silver.withColumn(
        "text",
        concat_ws(" ", coalesce(col("title"), lit("")), coalesce(col("content"), lit("")))
    )
else:
    df_news_silver = df_news_silver.withColumn(
        "text",
        coalesce(col("title"), lit(""))
    )

# Filter valid records
df_news_silver = df_news_silver.filter(
    col("created_ts").isNotNull() &
    col("title").isNotNull() &
    (length(col("text")) > 0)
).withColumn("title_hash", hash(col("title")))

# Remove duplicates
window_spec = Window.partitionBy("title_hash").orderBy("created_ts")
df_news_silver = df_news_silver.withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num", "title_hash")

print(f"\nCleaned articles: {df_news_silver.count()}")
df_news_silver.agg(min("date").alias("min_date"), max("date").alias("max_date")).show()


In [ ]:
# Clean price data - handle two different formats
# Format 1: Short column names (c, h, l, o, t, v)
# Format 2: Full column names (close, high, low, open, timestamp, volume) or different structure

price_columns = df_price_bronze.columns
print(f"Available price columns: {price_columns}")

# Helper function to find column by possible names
def find_col_name(cols, possible_names):
    for name in possible_names:
        if name in cols:
            return name
    return None

# Detect format and map columns
if "c" in price_columns and "h" in price_columns:
    # Format 1: Short names (c, h, l, o, t, v)
    df_price_silver = df_price_bronze.select(
        col("c").alias("close"),
        col("h").alias("high"),
        col("l").alias("low"),
        col("o").alias("open"),
        col("t").alias("timestamp"),
        col("v").alias("volume")
    )
elif "close" in price_columns and "high" in price_columns:
    # Format 2: Full names (close, high, low, open, timestamp/date, volume)
    timestamp_name = "timestamp" if "timestamp" in price_columns else "date"
    df_price_silver = df_price_bronze.select(
        col("close"),
        col("high"),
        col("low"),
        col("open"),
        col(timestamp_name).alias("timestamp"),
        col("volume")
    )
else:
    # Format 3: Try to infer column names
    close_name = find_col_name(price_columns, ['close', 'c', 'price', 'closing_price'])
    high_name = find_col_name(price_columns, ['high', 'h', 'high_price'])
    low_name = find_col_name(price_columns, ['low', 'l', 'low_price'])
    open_name = find_col_name(price_columns, ['open', 'o', 'opening_price'])
    timestamp_name = find_col_name(price_columns, ['timestamp', 't', 'time', 'date', 'datetime'])
    volume_name = find_col_name(price_columns, ['volume', 'v', 'vol'])
    
    select_cols = []
    if close_name:
        select_cols.append(col(close_name).alias("close"))
    if high_name:
        select_cols.append(col(high_name).alias("high"))
    if low_name:
        select_cols.append(col(low_name).alias("low"))
    if open_name:
        select_cols.append(col(open_name).alias("open"))
    if timestamp_name:
        select_cols.append(col(timestamp_name).alias("timestamp"))
    if volume_name:
        select_cols.append(col(volume_name).alias("volume"))
    
    df_price_silver = df_price_bronze.select(*select_cols)

# Handle timestamp conversion for both formats
# Format 1: Unix timestamp (numeric - milliseconds or seconds)
# Format 2: ISO timestamp string (e.g., '2016-01-04T05:00:00Z')

# Use a UDF to safely convert timestamps without casting errors
from datetime import datetime, timezone
from pyspark.sql.functions import udf
from pyspark.sql.types import TimestampType

def safe_timestamp_convert(ts):
    """Safely convert timestamp string to datetime"""
    if ts is None:
        return None
    ts_str = str(ts)
    
    # Try ISO format first
    if 'T' in ts_str or (ts_str.count('-') >= 2 and len(ts_str) > 10):
        formats = [
            "%Y-%m-%dT%H:%M:%SZ",
            "%Y-%m-%dT%H:%M:%S.%fZ",
            "%Y-%m-%dT%H:%M:%S",
            "%Y-%m-%d"
        ]
        for fmt in formats:
            try:
                dt = datetime.strptime(ts_str, fmt)
                return dt
            except:
                continue
    
    # Try numeric (Unix timestamp)
    try:
        ts_num = float(ts_str)
        if ts_num > 1e12:  # milliseconds
            return datetime.fromtimestamp(ts_num / 1000, tz=timezone.utc)
        else:  # seconds
            return datetime.fromtimestamp(ts_num, tz=timezone.utc)
    except:
        return None

# Register UDF
safe_timestamp_udf = udf(safe_timestamp_convert, TimestampType())

# Apply conversion
df_price_silver = df_price_silver.withColumn(
    "datetime",
    safe_timestamp_udf(col("timestamp"))
).withColumn("date", to_date(col("datetime")))

# Filter valid records
df_price_silver = df_price_silver.filter(
    col("close").isNotNull() & col("date").isNotNull()
).orderBy("date")

print(f"\nPrice records: {df_price_silver.count()}")
df_price_silver.select("date", "open", "high", "low", "close", "volume").show(5)


## 2.5. Create Sample Datasets

Create variables for full data and 20% sample for testing/development.


In [ ]:
# Create full data variables (lazy references - no extra actions)
df_news_silver_full = df_news_silver
df_price_silver_full = df_price_silver

# Create 20% sample for development/testing (lazy transformations)
df_news_silver_sample = df_news_silver.sample(fraction=0.2, seed=42)
df_price_silver_sample = df_price_silver.sample(fraction=0.2, seed=42)

# By default, use the 20% sample for downstream analysis
# (Switch to *_full variables later if you want full dataset)
df_news_silver = df_news_silver_sample
df_price_silver = df_price_silver_sample

print("Using 20% sample datasets for analysis. Switch to *_full variables for full run if needed.")


## 3. MLlib Sentiment Analysis


In [ ]:
# Financial sentiment lexicon (positive and negative words)
positive_words = [
    'gain', 'gains', 'gained', 'gaining', 'growth', 'grow', 'growing', 'rose', 'rises', 'rising', 'rise',
    'up', 'upside', 'upward', 'surge', 'surged', 'surges', 'surging', 'rally', 'rallied', 'rallies',
    'strong', 'stronger', 'strength', 'strengthen', 'strengthened', 'beat', 'beats', 'beating', 'beat',
    'profit', 'profits', 'profitable', 'profitability', 'earnings', 'revenue', 'revenues', 'increase',
    'increased', 'increasing', 'increases', 'boost', 'boosted', 'boosts', 'boosted', 'bullish', 'bull',
    'optimistic', 'optimism', 'positive', 'outperform', 'outperformed', 'outperforming', 'success',
    'successful', 'win', 'wins', 'winning', 'breakthrough', 'breakthroughs', 'momentum', 'soar', 'soared',
    'soaring', 'soars', 'jump', 'jumped', 'jumping', 'jumps', 'climb', 'climbed', 'climbing', 'climbs',
    'advance', 'advanced', 'advancing', 'advances', 'improve', 'improved', 'improving', 'improvement',
    'exceed', 'exceeded', 'exceeding', 'exceeds', 'exceeded', 'outpace', 'outpaced', 'outpacing'
]

negative_words = [
    'loss', 'losses', 'lost', 'losing', 'decline', 'declined', 'declines', 'declining', 'fall', 'fell',
    'falls', 'falling', 'down', 'downside', 'downward', 'drop', 'dropped', 'drops', 'dropping', 'plunge',
    'plunged', 'plunges', 'plunging', 'crash', 'crashed', 'crashes', 'crashing', 'weak', 'weaker',
    'weakness', 'weaken', 'weakened', 'miss', 'missed', 'missing', 'misses', 'disappoint', 'disappointed',
    'disappointing', 'disappointment', 'decrease', 'decreased', 'decreasing', 'decreases', 'cut', 'cuts',
    'cutting', 'reduce', 'reduced', 'reducing', 'reduces', 'reduction', 'bearish', 'bear', 'pessimistic',
    'pessimism', 'negative', 'underperform', 'underperformed', 'underperforming', 'failure', 'fail',
    'failed', 'failing', 'fails', 'tumble', 'tumbled', 'tumbles', 'tumbling', 'slump', 'slumped',
    'slumps', 'slumping', 'sink', 'sank', 'sinking', 'sinks', 'retreat', 'retreated', 'retreating',
    'retreats', 'worsen', 'worsened', 'worsening', 'worsens', 'deteriorate', 'deteriorated', 'deteriorating',
    'deteriorates', 'concern', 'concerns', 'concerned', 'worry', 'worries', 'worried', 'worrisome',
    'risk', 'risks', 'risky', 'uncertainty', 'uncertain', 'volatile', 'volatility', 'trouble', 'troubles'
]

# Create broadcast variables for efficient lookups
positive_broadcast = spark.sparkContext.broadcast(set(positive_words))
negative_broadcast = spark.sparkContext.broadcast(set(negative_words))
print("Sentiment lexicon loaded")


In [ ]:
# Define sentiment scoring function using lexicon
def calculate_sentiment(words):
    if not words or len(words) == 0:
        return (0.0, 0.0, 0.0, 0.0, 0.0)
    
    pos_set = positive_broadcast.value
    neg_set = negative_broadcast.value
    
    pos_count = sum(1 for word in words if word.lower() in pos_set)
    neg_count = sum(1 for word in words if word.lower() in neg_set)
    total_words = len(words)
    
    if total_words == 0:
        return (0.0, 1.0, 0.0, 0.0, 0.0)
    
    p_pos = pos_count / total_words if total_words > 0 else 0.0
    p_neg = neg_count / total_words if total_words > 0 else 0.0
    p_neu = 1.0 - p_pos - p_neg
    
    # Normalize probabilities
    total_prob = p_pos + p_neu + p_neg
    if total_prob > 0:
        p_pos = p_pos / total_prob
        p_neu = p_neu / total_prob
        p_neg = p_neg / total_prob
    
    sentiment_score = p_pos - p_neg
    confidence = max(p_pos, p_neu, p_neg)
    
    return (float(p_pos), float(p_neu), float(p_neg), float(sentiment_score), float(confidence))

# Create UDF
sentiment_schema = StructType([
    StructField("p_pos", DoubleType(), True),
    StructField("p_neu", DoubleType(), True),
    StructField("p_neg", DoubleType(), True),
    StructField("sentiment_score", DoubleType(), True),
    StructField("confidence", DoubleType(), True)
])
sentiment_udf = udf(calculate_sentiment, sentiment_schema)


In [ ]:
# Preprocess text using MLlib
tokenizer = Tokenizer(inputCol="text", outputCol="words")
stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

# Apply tokenization and stopword removal
df_news_silver = tokenizer.transform(df_news_silver)
df_news_silver = stopwords_remover.transform(df_news_silver)

# Apply sentiment analysis
print(f"Processing {df_news_silver.count()} articles...")
df_news_silver = df_news_silver.withColumn("sentiment_result", sentiment_udf(col("filtered_words"))) \
    .select("*",
        col("sentiment_result.p_pos").alias("p_pos"),
        col("sentiment_result.p_neu").alias("p_neu"),
        col("sentiment_result.p_neg").alias("p_neg"),
        col("sentiment_result.sentiment_score").alias("sentiment_score"),
        col("sentiment_result.confidence").alias("confidence")
    ).drop("sentiment_result", "words", "filtered_words")
print("Sentiment analysis complete")


In [ ]:
# Sentiment statistics
df_news_silver.select(
    mean("sentiment_score").alias("mean_sentiment"),
    stddev("sentiment_score").alias("std_sentiment"),
    mean("confidence").alias("mean_confidence")
).show()


## 4. Daily Aggregation (Gold Layer)


In [ ]:
# Aggregate sentiment by date
df_daily_sentiment = df_news_silver.groupBy("date").agg(
    mean("sentiment_score").alias("sent_mean"),
    stddev("sentiment_score").alias("sent_std"),
    count("*").alias("doc_count"),
    (sum(when(col("p_pos") > col("p_neg"), 1).otherwise(0)) / count("*")).alias("pos_ratio"),
    mean("confidence").alias("conf_mean")
)

print(f"Daily sentiment: {df_daily_sentiment.count()} days")
df_daily_sentiment.orderBy("date").show(10)


In [ ]:
# Calculate price returns and volatility
window_spec = Window.orderBy("date")
df_price_silver = df_price_silver.withColumn("prev_close", lag("close", 1).over(window_spec)) \
    .withColumn("ret_1d", when(col("prev_close").isNotNull() & (col("prev_close") != 0),
                               log(col("close") / col("prev_close"))).otherwise(None)) \
    .withColumn("vol_1d", abs(col("ret_1d")))

df_daily_price = df_price_silver.groupBy("date").agg(
    first("open").alias("open"),
    max("high").alias("high"),
    min("low").alias("low"),
    last("close").alias("close"),
    sum("volume").alias("volume"),
    last("ret_1d").alias("ret_1d"),
    last("vol_1d").alias("vol_1d")
)

print("Daily price metrics ready (aggregated by date). Showing first 10 rows:")
df_daily_price.orderBy("date").show(10)


In [ ]:
# Merge sentiment and price data
df_gold = df_daily_sentiment.join(df_daily_price, on="date", how="inner").orderBy("date")

print("Gold layer ready (days with both sentiment and price data). Showing first 10 rows:")
df_gold.select("date", "sent_mean", "doc_count", "close", "ret_1d", "vol_1d").show(10)


## 5. Correlation Analysis


In [ ]:
# Create a small, narrow sample for correlation and plotting to avoid large transfers
sentiment_vars = ['sent_mean', 'pos_ratio', 'doc_count']
market_vars = ['ret_1d', 'vol_1d']

df_gold_small = df_gold.select(["date"] + sentiment_vars + market_vars).sample(fraction=0.2, seed=42)

# Convert sampled data to Pandas for correlation analysis and visualization
df_gold_pd = df_gold_small.toPandas()
df_gold_pd['date'] = pd.to_datetime(df_gold_pd['date'])
print(f"Data used for correlation/plots: {len(df_gold_pd)} rows")


In [ ]:
# Calculate correlations on sampled data to keep results small
correlation_results = []
for sent_var in sentiment_vars:
    for market_var in market_vars:
        valid_data = df_gold_pd[[sent_var, market_var]].dropna()
        
        if len(valid_data) > 10:
            pearson_corr, pearson_p = stats.pearsonr(valid_data[sent_var], valid_data[market_var])
            spearman_corr, spearman_p = stats.spearmanr(valid_data[sent_var], valid_data[market_var])
            
            correlation_results.append({
                'sentiment_metric': sent_var,
                'market_metric': market_var,
                'pearson_corr': pearson_corr,
                'pearson_p': pearson_p,
                'spearman_corr': spearman_corr,
                'spearman_p': spearman_p
            })

df_correlations = pd.DataFrame(correlation_results)
print("Correlation Results (sampled data):")
for _, row in df_correlations.iterrows():
    print(f"{row['sentiment_metric']} vs {row['market_metric']}: "
          f"Pearson={row['pearson_corr']:.4f} (p={row['pearson_p']:.4f}), "
          f"Spearman={row['spearman_corr']:.4f} (p={row['spearman_p']:.4f})")


In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
corr_matrix = df_gold_pd[sentiment_vars + market_vars].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Matrix: Sentiment vs Market Metrics')
plt.tight_layout()
plt.show()


In [ ]:
# Time series plots
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(df_gold_pd['date'], df_gold_pd['sent_mean'], 'b-', alpha=0.7, linewidth=1.5)
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[0].set_ylabel('Sentiment Score')
axes[0].set_title('Daily Mean Sentiment Score')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_gold_pd['date'], df_gold_pd['ret_1d'], 'g-', alpha=0.7, linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[1].set_ylabel('Daily Returns')
axes[1].set_title('AAPL Daily Returns')
axes[1].grid(True, alpha=0.3)

axes[2].plot(df_gold_pd['date'], df_gold_pd['vol_1d'], 'r-', alpha=0.7, linewidth=1.5)
axes[2].set_ylabel('Volatility')
axes[2].set_xlabel('Date')
axes[2].set_title('AAPL Daily Volatility')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Scatter plots with trend lines
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

def plot_scatter(ax, x, y, title, xlabel, ylabel, color='blue'):
    ax.scatter(x, y, alpha=0.5, s=30, color=color)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if len(x.dropna()) > 1:
        z = np.polyfit(x.dropna(), y.dropna(), 1)
        p = np.poly1d(z)
        ax.plot(x.dropna(), p(x.dropna()), "r--", alpha=0.8, linewidth=2)

plot_scatter(axes[0, 0], df_gold_pd['sent_mean'], df_gold_pd['ret_1d'], 
             'Sentiment vs Returns', 'Sentiment Score', 'Daily Returns')
plot_scatter(axes[0, 1], df_gold_pd['sent_mean'], df_gold_pd['vol_1d'], 
             'Sentiment vs Volatility', 'Sentiment Score', 'Volatility', 'orange')
plot_scatter(axes[1, 0], df_gold_pd['pos_ratio'], df_gold_pd['ret_1d'], 
             'Positive Ratio vs Returns', 'Positive Ratio', 'Daily Returns', 'green')
plot_scatter(axes[1, 1], df_gold_pd['doc_count'], df_gold_pd['vol_1d'], 
             'News Volume vs Volatility', 'Document Count', 'Volatility', 'purple')

plt.tight_layout()
plt.show()


In [ ]:
# Overlay: Sentiment and Returns
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.set_xlabel('Date')
ax1.set_ylabel('Sentiment Score', color='tab:blue')
ax1.plot(df_gold_pd['date'], df_gold_pd['sent_mean'], color='tab:blue', alpha=0.7, linewidth=2)
ax1.axhline(y=0, color='tab:blue', linestyle='--', alpha=0.3)
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.set_ylabel('Daily Returns', color='tab:green')
ax2.plot(df_gold_pd['date'], df_gold_pd['ret_1d'], color='tab:green', alpha=0.7, linewidth=2)
ax2.axhline(y=0, color='tab:green', linestyle='--', alpha=0.3)
ax2.tick_params(axis='y', labelcolor='tab:green')

plt.title('Sentiment and Returns Co-movement')
fig.tight_layout()
plt.show()


In [ ]:
# Summary statistics
print("Sentiment Metrics:")
df_gold.select(
    mean("sent_mean").alias("mean"),
    stddev("sent_mean").alias("std"),
    mean("pos_ratio").alias("pos_ratio"),
    mean("doc_count").alias("doc_count")
).show()

print("\nMarket Metrics:")
df_gold.select(
    mean("close").alias("mean_close"),
    mean("ret_1d").alias("mean_return"),
    stddev("ret_1d").alias("std_return"),
    mean("vol_1d").alias("mean_volatility")
).show()

print("\nData Coverage:")
df_gold.agg(
    count("*").alias("total_days"),
    min("date").alias("min_date"),
    max("date").alias("max_date"),
    mean("doc_count").alias("avg_articles_per_day")
).show()


## 6. Optional: Save Results

```python
# Save to Delta Lake (if using Databricks)
# df_gold.write.format("delta").mode("overwrite").saveAsTable("gold.sentiment_price_analysis")
```


## 7. Summary

This analysis implements a Bronze → Silver → Gold data pipeline using PySpark and MLlib:
- **Bronze**: Raw JSON data ingestion
- **Silver**: Data cleaning and MLlib-based sentiment analysis using financial lexicon
- **Gold**: Daily aggregation and correlation with price data

Key findings show correlations between sentiment metrics and market indicators, visualized through time series and scatter plots.


In [ ]:
# spark.stop()  # Uncomment to stop Spark session
print("Analysis complete!")
